## Aggregations a gold

### Objetivo:
    - Obtener informacion sobre "total de presupuesto" y "total de ingresos" de "peliculas", el pais donde se realizo la grabacion y cual fue la productora
    - A partir de fecha de lanzamiento 2015 
    - agrupar por "año d el afecha de lanzamiento" y "genero" 
    - ranking ordenado de manera ascendente por el "total del presupuesto" y "total de ingresos" particionado por el año de lanzamiento


tables: movie, genre, movie_genre \
columns: year_release_date, budget, revenue, genre_name, release_date \
add: rank 

df -> results_group_movie_genre

### Leer archivos

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date =dbutils.widgets.get("p_file_date")

In [0]:
# %run "../includes/configuration" 

In [0]:
# movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df = spark.read.table("movie_silver.movies") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
# genres_df = spark.read.parquet(f"{silver_folder_path}/genre")
genres_df = spark.read.table("movie_silver.genre")

In [0]:
# movies_genres_df = spark.read.parquet(f"{silver_folder_path}/movies_genres")
movies_genres_df = spark.read.table("movie_silver.movies_genres") \
                    .filter(f"file_date = '{v_file_date}'")

### Join "genre" y "movies_genres"

In [0]:
genres_mov_gen_df = genres_df.join(movies_genres_df,
                                            genres_df.genre_id == movies_genres_df.genre_id,
                                            "inner") \
                                        .select(genres_df.genre_name,movies_genres_df.movie_id)

### Join "genres_movie" y "movies_df"

#### - filtrar las peliculas por fecha de lanzamiento mayor o igual a 2015

In [0]:
movies_filtered_df = movies_df.filter("year_release_date >=  2015")

In [0]:
results_movies_genres_df = movies_filtered_df.join(genres_mov_gen_df,
                                                    movies_filtered_df.movie_id == genres_mov_gen_df.movie_id,
                                                    "inner")

In [0]:
results_df = results_movies_genres_df \
    .select("year_release_date", "genre_name", "budget", "revenue")

In [0]:
from pyspark.sql.functions import sum, desc, dense_rank, asc, lit
from pyspark.sql.window import Window

In [0]:
results_order_by_df = results_df \
    .groupBy("year_release_date", "genre_name") \
    .agg(
        sum("budget").alias("total_budget"),
        sum("revenue").alias("total_revenue")
    )

In [0]:
display(results_order_by_df)

#### - Agregar el ranking

In [0]:
results_dense_rank_df = Window.partitionBy("year_release_date").orderBy(
                                                                desc("total_budget"), 
                                                                desc("total_revenue"))
final_df = results_order_by_df.withColumn("dense_rank", dense_rank().over(results_dense_rank_df)) \
                                .withColumn("created_date", lit(v_file_date))

### Escribir los datos en el datalake en formato "Parquet"

In [0]:
# overwrite_partition("movie_gold", "results_group_movie_genre","created_date",v_file_date)

In [0]:
# final_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_group_movie_genre")
# final_df.write.mode("overwrite").format("delta").saveAsTable("movie_gold.results_group_movie_genre")
# final_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_group_movie_genre")

In [0]:
# hago una validacion con merge

merge_condition = 'tgt.year_release_date = src.year_release_date AND tgt.genre_name = src.genre_name AND tgt.created_date=src.created_date'
merge_delta_lake(final_df, "movie_gold", "results_group_movie_genre", merge_condition, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_genre

In [0]:
# display(spark.read.parquet(f"{gold_folder_path}/results_group_movie_genre"))